# EDA — Macro Panel
Source: `data/macro_panel_data_raw.xlsx` cleaned by `data/clean_macro_panel.py`  
33 Bloomberg series across 7 groups: **FX, Rates, Risk, Energy, Inflation, Growth, Supply**  
Mix of daily (D), weekly (W), monthly (M) frequency. Raw levels only — transforms applied per series.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

_cwd = Path(".").resolve()
REPO_ROOT  = _cwd if (_cwd / "src").exists() else _cwd.parent
MACRO_DIR  = REPO_ROOT / "data" / "macro"
SERIES_DIR = MACRO_DIR / "series"

assert MACRO_DIR.exists(), f"Run data/clean_macro_panel.py first — {MACRO_DIR} not found"

manifest = pd.read_csv(MACRO_DIR / "manifest.csv", parse_dates=["start", "end"])
wide     = pd.read_csv(MACRO_DIR / "macro_panel_raw.csv", index_col="date", parse_dates=True)

GROUP_ORDER = ["FX", "Rates", "Risk", "Energy", "Inflation", "Growth", "Supply"]
GROUP_COLORS = {
    "FX": "steelblue", "Rates": "darkorange", "Risk": "crimson",
    "Energy": "green", "Inflation": "purple", "Growth": "teal", "Supply": "saddlebrown",
}
group_order_map = {g: i for i, g in enumerate(GROUP_ORDER)}
group_of        = manifest.set_index("name")["group"]
included        = manifest[manifest["include"]].copy()

MARKET_EVENTS = {
    "COVID crash":    (pd.Timestamp("2020-03-20"), "red"),
    "Russia-Ukraine": (pd.Timestamp("2022-02-24"), "darkorange"),
    "Fed hike start": (pd.Timestamp("2022-03-16"), "purple"),
}

print(f"Manifest: {len(manifest)} series ({manifest['include'].sum()} included, {(~manifest['include']).sum()} excluded)")
print(f"Wide panel: {wide.shape[0]} dates ({wide.index.min().date()} – {wide.index.max().date()}), {wide.shape[1]} series")

## 1. Series Manifest

In [ ]:
display(
    manifest[["name", "group", "freq", "transform", "start", "end", "n_obs", "include", "circular", "note"]]
    .sort_values(["group", "name"])
    .reset_index(drop=True)
)

## 2. Data Coverage Heatmap
Green = data available on that date, white = missing.

In [ ]:
monthly_coverage = wide.resample("ME").count()
has_data = (monthly_coverage > 0).astype(int)

col_order = sorted(
    [c for c in wide.columns if c in group_of.index],
    key=lambda n: (group_order_map.get(group_of[n], 99), n)
)
has_data = has_data[[c for c in col_order if c in has_data.columns]]

fig, ax = plt.subplots(figsize=(20, 6))
sns.heatmap(has_data.T, cmap="Greens", cbar=False, ax=ax, linewidths=0, xticklabels=False)
ax.set_xlabel("Date (monthly)")
ax.set_ylabel("Series")
ax.set_title("Data coverage by series and month (green = available)")
ax.tick_params(axis="y", labelsize=7)
years = has_data.index.year.unique()
year_ticks = [has_data.index.get_loc(has_data[has_data.index.year == y].index[0]) for y in years if y in has_data.index.year]
ax.set_xticks(year_ticks)
ax.set_xticklabels([str(y) for y in years if y in has_data.index.year], rotation=45, fontsize=8)
fig.tight_layout()
plt.show()

## 3. Missing Data Summary

In [ ]:
missing = pd.DataFrame({
    "n_missing": wide.isna().sum(),
    "pct_missing": (wide.isna().mean() * 100).round(1),
    "n_obs": wide.notna().sum(),
}).join(manifest.set_index("name")[["group", "freq", "transform"]])
missing = missing.sort_values("pct_missing", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [GROUP_COLORS.get(g, "gray") for g in missing["group"]]
ax.barh(missing.index, missing["pct_missing"], color=colors, alpha=0.8)
ax.set_xlabel("% missing")
ax.set_title("Missing data % per series (colour = group)")
ax.axvline(5, color="red", lw=1, ls="--", alpha=0.5)
ax.tick_params(axis="y", labelsize=8)
# Legend
from matplotlib.patches import Patch
handles = [Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=handles, fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

display(missing.reset_index().rename(columns={"index": "name"}))

## 4. Time Series by Group

In [ ]:
for group in GROUP_ORDER:
    series_names = included[included["group"] == group]["name"].tolist()
    if not series_names:
        continue
    n = len(series_names)
    fig, axes = plt.subplots(n, 1, figsize=(14, 2.5 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, name in zip(axes, series_names):
        s = wide[name].dropna()
        meta = manifest.set_index("name").loc[name]
        ax.plot(s.index, s.values, lw=0.9, color=GROUP_COLORS[group])
        ax.set_title(f"{name}  [{meta['transform']} | {meta['freq']}]", fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        for ev, (ev_date, ev_col) in MARKET_EVENTS.items():
            if s.index.min() <= ev_date <= s.index.max():
                ax.axvline(ev_date, color=ev_col, lw=1.2, ls="--", alpha=0.8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.suptitle(f"Group: {group}", fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.show()

## 5. Distribution Summary (Included Daily Series)
Boxplots of z-scored values — highlights outliers and fat tails across series.

In [ ]:
daily_names = included[included["freq"] == "D"]["name"].tolist()
daily = wide[daily_names].copy()
daily_z = (daily - daily.mean()) / daily.std()

daily_names_sorted = sorted(daily_names, key=lambda n: (group_order_map.get(group_of.get(n, ""), 99), n))
daily_z = daily_z[daily_names_sorted]

fig, ax = plt.subplots(figsize=(16, 5))
bp = ax.boxplot(
    [daily_z[c].dropna().values for c in daily_names_sorted],
    labels=daily_names_sorted, patch_artist=True, flierprops={"markersize": 2, "alpha": 0.3},
    medianprops={"color": "black", "lw": 1.5},
)
for patch, name in zip(bp["boxes"], daily_names_sorted):
    patch.set_facecolor(GROUP_COLORS.get(group_of.get(name, ""), "gray"))
    patch.set_alpha(0.6)
ax.axhline(0, color="k", lw=0.5, ls="--")
ax.set_ylabel("z-score")
ax.set_title("Distribution of z-scored daily series (colour = group)")
ax.tick_params(axis="x", rotation=90, labelsize=8)
handles = [Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=handles, fontsize=8)
fig.tight_layout()
plt.show()

## 6. Descriptive Statistics

In [ ]:
stats = wide[included["name"].tolist()].describe().T
stats["skew"]     = wide[included["name"].tolist()].skew()
stats["kurtosis"] = wide[included["name"].tolist()].kurtosis()
stats = stats.join(manifest.set_index("name")[["group", "freq", "transform"]])
stats = stats.sort_values(["group", "name"] if "name" in stats.columns else "group")
display(stats.round(4))

## 7. Correlation Heatmap (Daily Series)
Pairwise Pearson correlation on raw levels, daily frequency.

In [ ]:
corr = daily[daily_names_sorted].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 7},
    square=True, cbar_kws={"label": "Pearson r", "shrink": 0.6}, ax=ax,
)
ax.set_title("Correlation — daily macro series (raw levels)")
ax.tick_params(axis="both", labelsize=8)
fig.tight_layout()
plt.show()

## 8. Rolling Volatility (Daily Series)
63-day rolling standard deviation — shows regime changes in macro volatility.

In [ ]:
ROLL = 63

for group in GROUP_ORDER:
    names = [n for n in daily_names_sorted if group_of.get(n) == group]
    if not names:
        continue
    fig, ax = plt.subplots(figsize=(14, 3))
    for name in names:
        s = wide[name].dropna()
        ax.plot(s.rolling(ROLL).std(), lw=0.9, alpha=0.8, label=name)
    for ev, (ev_date, ev_col) in MARKET_EVENTS.items():
        ax.axvline(ev_date, color=ev_col, lw=1.2, ls="--", alpha=0.7, label=ev)
    ax.set_title(f"{group} — {ROLL}-day rolling std")
    ax.set_ylabel("Rolling std")
    ax.legend(fontsize=7, ncol=4)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()

## 9. Cross-group Correlation: Macro vs Risk Sentiment
Correlation of each series against VIX and HY OAS — key stress indicators.

In [ ]:
anchors = ["vix", "credit_hy_oas"]
targets = [n for n in daily_names_sorted if n not in anchors]

corr_vs_risk = pd.DataFrame({
    a: [daily[[a, t]].dropna().corr().iloc[0, 1] for t in targets]
    for a in anchors
}, index=targets)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, anchor in zip(axes, anchors):
    vals = corr_vs_risk[anchor].sort_values()
    colors = [GROUP_COLORS.get(group_of.get(n, ""), "gray") for n in vals.index]
    ax.barh(vals.index, vals.values, color=colors, alpha=0.8)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(f"Correlation vs {anchor}")
    ax.set_xlabel("Pearson r")
    ax.tick_params(axis="y", labelsize=8)
handles = [Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
axes[1].legend(handles=handles, fontsize=8, loc="lower right")
fig.suptitle("Cross-group correlation vs risk sentiment anchors", fontsize=12)
fig.tight_layout()
plt.show()

display(corr_vs_risk.round(3))

## 10. Monthly Series Overview
Forward-filled to daily for plotting; dots mark actual observation dates.

In [ ]:
monthly_names = included[included["freq"].isin(["M", "W"])]["name"].tolist()
n = len(monthly_names)
fig, axes = plt.subplots(n, 1, figsize=(14, 2.8 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, name in zip(axes, monthly_names):
    s = wide[name].dropna()
    meta = manifest.set_index("name").loc[name]
    ax.plot(s.index, s.values, lw=0.8, color=GROUP_COLORS[meta["group"]], alpha=0.6)
    ax.scatter(s.index, s.values, s=6, color=GROUP_COLORS[meta["group"]], zorder=3)
    ax.set_title(f"{name}  [{meta['transform']} | {meta['freq']} | {meta['group']}]", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    for ev, (ev_date, ev_col) in MARKET_EVENTS.items():
        if s.index.min() <= ev_date <= s.index.max():
            ax.axvline(ev_date, color=ev_col, lw=1.1, ls="--", alpha=0.7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle("Monthly & weekly series (dots = actual observations)", fontsize=13)
fig.tight_layout()
plt.show()